In [ ]:
"""HITL case — LangGraph 1.2 `interrupt()` + `Command(resume=...)` 模式.

场景: Research Draft Editor
  draft_node     →  生成初稿
  human_review   →  `interrupt(...)` 暂停,等待人类裁决
  revise_node    →  根据反馈改稿,然后**再次**回到 human_review(回归循环)
  publish_node   →  终态:打印"已发布"并把 draft 写入 final_text

三种合法的人类回应:
  1. {"action": "approve"}              → 直接发布
  2. {"action": "revise", "feedback": ...} → 改稿,再回到人类审
  3. {"action": "rewrite", "text": ...}    → 人类直接给出终稿,跳过改稿

Notes (重要!):
  - LangGraph 1.2 起,`interrupt` 抛出的不再是异常,而是把整张图冻结在
    `__interrupt__` 状态,`graph.invoke(...)` 返回的就是包含 `__interrupt__` 的
    状态字典(不是抛异常)。
  - 恢复用 `Command(resume=<payload>)`,payload 即 `interrupt()` 收到的参数。
  - 想跳到不同节点请走**条件边**(conditional edge)+ 状态字段,不要把
    `Command(goto=...)` 和 `LastValue` reducer 混着用 —— 重入图时会冲突。
  - 本例用 `MemorySaver`(内存 checkpointer)。生产请换 PostgresSaver,见
    src/apps/deep_research/graph.py 的 build_graph 用法。
"""

import operator
import uuid
from typing import Annotated, Literal, TypedDict

import dotenv
from IPython.display import Image, display
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from rich import print as rprint

dotenv.load_dotenv(override=True)

In [ ]:
class ResearchDraftState(TypedDict):
    topic: str                          # 调研主题
    draft: str                          # 当前稿(可被 revise_node 改写)
    review_action: str                  # 'approve' | 'revise' | 'rewrite'
    feedback: str                       # revise 时的反馈意见
    final_text: str                     # publish_node 才写入
    revision_round: Annotated[int, operator.add]   # 记录改了 N 次(用 operator.add 自动 +1)
    messages: Annotated[list[str], operator.add]   # 记录人类/节点互动日志

In [ ]:
def draft_node(state: ResearchDraftState) -> dict:
    """生成(初稿 或 第 N+1 稿),把当前轮次写入 messages."""
    round_no = state.get("revision_round", 0) + 1
    feedback = state.get("feedback")
    if feedback:
        draft = (
            f"[第 {round_no} 稿] 主题: {state['topic']}\n"
            f"基于反馈({feedback})已重写,补充了数据引用与对比分析。"
        )
    else:
        draft = (
            f"[初稿] 主题: {state['topic']}\n"
            "本章先给出现状描述,后续章节将深入分析关键驱动因素。"
        )
    return {
        "draft": draft,
        "revision_round": 1,  # operator.add reducer: 1 表示这一轮计 1 次
        "messages": [f"draft_node: round={round_no}, len(draft)={len(draft)}"],
    }


def human_review_node(state: ResearchDraftState) -> dict:
    """关键 HITL 节点 —— `interrupt()` 会冻结整张图.

    payload 给人类三件事:当前稿、上一轮反馈、当前是第几轮。
    人类回复通过 `Command(resume=<payload>)` 回流。
    """
    payload = {
        "question": f"请审阅第 {state.get('revision_round', 0) + 1} 稿",
        "draft": state["draft"],
        "round": state.get("revision_round", 0),
        "previous_feedback": state.get("feedback"),
        "options": ["approve", "revise: ...", "rewrite: ..."],
    }

    # interrupt 返回值 == human 给回的 resume payload(LangGraph 1.2 标准行为)
    decision = interrupt(payload)

    # 规整化人类回应(允许 str 或 dict)
    if isinstance(decision, str):
        decision = {"action": decision}
    action = decision.get("action", "revise")
    feedback = decision.get("feedback", "")
    rewritten = decision.get("text", "")

    update: dict = {
        "review_action": action,
        "feedback": "",        # 默认清空,后续节点按需填充
        "messages": [f"human_review: action={action}"],
    }
    if action == "revise":
        update["feedback"] = feedback or "请补充具体例子"
    elif action == "rewrite":
        # 人类直接给出终稿 —— 把 draft 覆盖掉,下一步直接 publish
        update["draft"] = rewritten or state["draft"]
    elif action == "approve":
        pass  # 啥都不改,让条件边把它送往 publish
    else:
        raise ValueError(f"未知 action: {action!r}")
    return update


def publish_node(state: ResearchDraftState) -> dict:
    """终态:把 draft 定稿到 final_text,并打一行日志."""
    return {
        "final_text": state["draft"],
        "messages": [f"publish: draft 长度={len(state['draft'])} 进入 final_text"],
    }


# ---- 条件边:决定 human_review_node 之后去哪 ----
def route_after_review(state: ResearchDraftState) -> Literal["revise_node", "publish_node"]:
    if state.get("review_action") == "approve":
        return "publish_node"
    if state.get("review_action") == "rewrite":
        return "publish_node"  # rewrite 的终稿已经是 final,直接 publish
    return "revise_node"


g = StateGraph(ResearchDraftState)
g.add_node("draft_node", draft_node)
g.add_node("human_review_node", human_review_node)
g.add_node("revise_node", draft_node)   # 重用 draft_node 函数,简化图;改成单独的 revise_node 也行
g.add_node("publish_node", publish_node)

g.add_edge(START, "draft_node")
g.add_edge("draft_node", "human_review_node")
g.add_conditional_edges(
    "human_review_node",
    route_after_review,
    {"revise_node": "revise_node", "publish_node": "publish_node"},
)
g.add_edge("revise_node", "human_review_node")   # 改完再回人审 —— 闭环
g.add_edge("publish_node", END)

# 内存 checkpointer,够 case 用;生产换 PostgresSaver
memory = InMemorySaver()
graph = g.compile(checkpointer=memory)

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# ---------- Scenario 1: 一次 approve ----------
tid = f"hitl-s1-{uuid.uuid4()}"
cfg = {"configurable": {"thread_id": tid}}

res1 = graph.invoke(
    {"topic": "酒店旅游业 AI Agent 落地", "draft": "", "review_action": "",
     "feedback": "", "final_text": "", "revision_round": 0, "messages": []},
    config=cfg,
)
rprint("首次 invoke 命中 interrupt →")
rprint({k: res1[k] for k in ("draft", "revision_round", "messages")})
rprint("__interrupt__ payload:", res1["__interrupt__"][0].value)

In [ ]:
res1b = graph.invoke(Command(resume={"action": "approve"}), config=cfg)
rprint("approve 后:", "final_text=", res1b["final_text"])
rprint("全部 messages:", res1b["messages"])
assert res1b["final_text"], "expected draft 已 publish 到 final_text"
# approve 后,LangGraph 把 `__interrupt__` 键整个从返回里省掉了(没有 pending 才省略)。
# 用 snap.next 看下一个待跑节点:走完图后应为 ()。
assert graph.get_state(cfg).next == ()

In [ ]:
# ---------- Scenario 2: revise → 再回人审 → approve (同一 thread_id) ----------
tid2 = f"hitl-s2-{uuid.uuid4()}"
cfg2 = {"configurable": {"thread_id": tid2}}

graph.invoke(
    {"topic": "AI Agent 在客服场景的 ROI", "draft": "", "review_action": "",
     "feedback": "", "final_text": "", "revision_round": 0, "messages": []},
    config=cfg2,
)
# 第一轮:人类要求修改,带上具体反馈
res2a = graph.invoke(
    Command(resume={"action": "revise", "feedback": "请加上 2 个具体酒店案例"}),
    config=cfg2,
)
rprint("改完一稿后再次进入 interrupt →")
rprint("draft:", res2a["draft"])
rprint("revision_round:", res2a["revision_round"])

In [ ]:
# 第二轮:人类最终 approve
res2b = graph.invoke(Command(resume={"action": "approve"}), config=cfg2)
rprint(res2b["messages"])
assert res2b["revision_round"] == 2, f"应有两轮 revise,got {res2b['revision_round']}"
assert res2b["final_text"]

In [ ]:
# ---------- Scenario 3: rewrite —— 人类直接给终稿(跳过 revise) ----------
tid3 = f"hitl-s3-{uuid.uuid4()}"
cfg3 = {"configurable": {"thread_id": tid3}}

graph.invoke(
    {"topic": "酒店旅游业 AI Agent 落地", "draft": "", "review_action": "",
     "feedback": "", "final_text": "", "revision_round": 0, "messages": []},
    config=cfg3,
)
res3 = graph.invoke(
    Command(resume={"action": "rewrite", "text": "[人类终稿] 直接采用,不需 revise。"}),
    config=cfg3,
)
rprint("rewrite 直达 final_text:", res3["final_text"])
assert res3["review_action"] == "rewrite"
assert res3["final_text"].startswith("[人类终稿]")

In [ ]:
# ---------- 旁路:用 graph.get_state() 看 checkpoint --------
snap = graph.get_state(cfg3)
rprint("next node:", snap.next)
rprint("values keys:", list(snap.values.keys()))
rprint("最后一条 message:", snap.values["messages"][-1])

## 几条容易踩坑的地方

1. **`interrupt()` 1.2 起不抛异常** —— `graph.invoke(...)` 返回 `dict['__interrupt__']`,
   不要写 `try/except` 去接,会被自己绕进去。

2. **`Command(resume=...)` 一定要配 checkpointer**。无 checkpointer 时,
   LangGraph 1.2 不会报错,但图一重启就丢失中断点 —— 看似能跑但实际上每条 `resume`
   都会变成新一轮的初始输入。本例用 `InMemorySaver`,生产请换成 PostgresSaver
   (项目里 `src/apps/deep_research/graph.py:build_graph` 就是同款用法)。

3. **`Command(goto=...)` 和普通 `LastValue`/条件边混用要小心**。一旦节点声明
   `goto`,后续 `Command` 跳到该节点时可能和原 `LastValue` reducer 撞上。
   **推荐做法**:把决策放进状态字段(本例的 `review_action`),用条件边路由。

4. **同一 `thread_id` 是命脉**。每个用户会话/每个文档编辑,都开一个新
   `thread_id`(`uuid.uuid4()`),否则状态串了别想排查。

5. **`operator.add` reducer** 用于像 `revision_round`、`messages` 这种
   "append 计数"字段;策划改稿轮次时不用自己维护累加。

6. **想保存 LLM 痕迹到 Langfuse**,把 `draft_node` 换成
   `langchain_openai.ChatOpenAI` 调用并配 `@observe`/Langfuse callback,
   payload 完全一致 —— interrupt 还是冻结在同一个 `human_review_node`。